# s1p36 analysis
Monthly category performance of an online store: what is happening, what to do, and what to watch. Every cell runs the same functions as `main.py`.

In [ ]:
import sys
from pathlib import Path
here = Path.cwd().resolve()
for p in [here, *here.parents]:
    if (p / "config" / "settings.py").exists():
        sys.path.insert(0, str(p))
        break
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import config.settings as S
from src.utils import io_utils as io
from src.utils.paths import FILES, ensure_dirs, chart_path
from src.data import load_raw as intake
from src.data import clean as cleaning
from src.features import derive as feats
from src.analysis import profiling as explore
from src.models import classifier as primary
from src.models import segmentation as segments
from src.models import projection as forecast
from src.analytics import alerts as rules
from src.analytics import scenarios as scenarios
from src.analytics import targets as targets
from src.analytics import insights as insights
from src.reporting import workbook as workbook
from src.reporting import dashboard
from IPython.display import Image, display
ensure_dirs()

## Data intake and validation
Read the raw file, match the column names, hide personal data and run domain checks.

In [ ]:
raw = intake.load_raw(verbose=True)
raw.head()

## Cleaning
Fix types and dates, drop duplicates and impossible rows, scale percentages, fill gaps and cap extreme ratios.

In [ ]:
clean = cleaning.clean(raw, verbose=True)
clean.describe().T.head(12)

## Feature building
Explainable measures: funnel rates, discount intensity, stockout share, margin, health score and the at-risk target.

In [ ]:
df = feats.build(clean, verbose=True)
df.head()

## Exploratory charts
Each chart answers one business question.

In [ ]:
if 'df' not in globals():
    df = io.read_frame(FILES['feature_store'])
specs = explore.run(df, verbose=True)
for key in ['target', 'attainment', 'cat_rate', 'trend']:
    display(Image(filename=str(chart_path(key))))

## Risk model
Three or more models compared by cross-validation on training rows, then scored once on held-out rows against a naive baseline.

In [ ]:
meta = primary.run(df, verbose=True)
display(Image(filename=str(chart_path('model_eval'))))
display(Image(filename=str(chart_path('importance'))))

## Segments and unusual months

In [ ]:
seg = segments.run(df, verbose=True)
display(Image(filename=str(chart_path('segments'))))

## Next-month revenue forecast
Chronological holdout, compared with a last-value baseline.

In [ ]:
fmeta = forecast.run(df, verbose=True)
display(Image(filename=str(chart_path('forecast_eval'))))

## Business rules and actions

In [ ]:
actions = rules.run(df, verbose=True)
display(Image(filename=str(chart_path('priority'))))
actions[actions['is_latest'] == 1].head(10)

## What-if scenarios

In [ ]:
sims = scenarios.run(df, verbose=True)

## Goals and alerts

In [ ]:
progress, alerts = targets.run(df, verbose=True)
pd.DataFrame(progress)

## Insights

In [ ]:
items = insights.run(df, verbose=True)

## Workbook and interactive page

In [ ]:
workbook.run(df, verbose=True)
try:
    dashboard.run(df, verbose=True)
except Exception as exc:
    print('Interactive page skipped:', exc)

## Summary

In [ ]:
C, F = S.C, S.F
print('=' * 60)
print('Rows:', len(df), '| Categories:', df[C['category']].nunique())
print('At-risk share: %.1f%%' % (100 * df[F['target']].mean()))
print('Net revenue: {:,.0f} against target {:,.0f}'.format(df[C['net_rev']].sum(), df[C['rev_target']].sum()))
print('=' * 60)